In [3]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from libpysal.weights import KNN
from esda import Moran
import numpy as np

# -------------------------------------------------------
# 1. Load and prepare data
# -------------------------------------------------------
fastfood = gpd.read_file("Data/fastfood_location_name_amenity.gpkg")
crashes = pd.read_csv("Data/CrashStatisticsFranklinCounty.csv")

# drop missing coords
crashes = crashes.dropna(subset=["Latitude", "Longitude"])

# create GeoDataFrames
gdf_crashes = gpd.GeoDataFrame(
    crashes,
    geometry=gpd.points_from_xy(crashes["Longitude"], crashes["Latitude"]),
    crs="EPSG:4326"
)
gdf_fastfood = fastfood.copy()

# ensure CRS match
if gdf_fastfood.crs.to_epsg() != 4326:
    gdf_fastfood = gdf_fastfood.to_crs(epsg=4326)

# -------------------------------------------------------
# 2. Compute distance to nearest fast-food outlet
# -------------------------------------------------------
# for computational efficiency, project to meters
gdf_crashes_m = gdf_crashes.to_crs(epsg=3857)
gdf_fastfood_m = gdf_fastfood.to_crs(epsg=3857)

# compute nearest fast-food distance for each crash
nearest_dist = []
for crash_point in gdf_crashes_m.geometry:
    dist = gdf_fastfood_m.distance(crash_point).min()
    nearest_dist.append(dist)

gdf_crashes_m["dist_to_fastfood_m"] = nearest_dist

# -------------------------------------------------------
# 3. Build spatial weights (neighbors)
# -------------------------------------------------------
# use KNN = 8 nearest crashes as neighborhood
w = KNN.from_dataframe(gdf_crashes_m, k=8)
w.transform = 'R'

# -------------------------------------------------------
# 4. Compute Moran’s I
# -------------------------------------------------------
# variable of interest: inverse distance (closer = higher value)
gdf_crashes_m["fastfood_proximity"] = 1 / (gdf_crashes_m["dist_to_fastfood_m"] + 1)

moran = Moran(gdf_crashes_m["fastfood_proximity"], w)

# -------------------------------------------------------
# 5. Print and interpret results
# -------------------------------------------------------
print("Moran’s I statistic:", moran.I)
print("Expected I (under null):", moran.EI)
print("Z-score:", moran.z)
print("p-value:", moran.p_sim)

if moran.p_sim < 0.05:
    print("✅ Significant spatial autocorrelation detected.")
else:
    print("❌ No significant spatial autocorrelation detected.")


C:\Users\raab.75\AppData\Local\Temp\ipykernel_31036\615560613.py:12: DtypeWarning: Columns (23,31) have mixed types. Specify dtype option on import or set low_memory=False.
  crashes = pd.read_csv("Data/CrashStatisticsFranklinCounty.csv")
c:\Users\raab.75\AppData\Local\miniconda3\envs\erdos_ds_environment\Lib\site-packages\libpysal\weights\distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 29 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


Moran’s I statistic: 0.4922646996895794
Expected I (under null): -8.70852564660803e-06
Z-score: [-0.02013265 -0.11814759 -0.11870933 ...  0.07910131 -0.06720588
 -0.11196529]
p-value: 0.001
✅ Significant spatial autocorrelation detected.


In [6]:
# -------------------------------------------------------
# 5. Visualization: Moran scatter plot (corrected)
# -------------------------------------------------------
from splot.esda import moran_scatterplot

fig, ax = moran_scatterplot(moran, aspect_equal=True)
ax.set_title("Moran’s I Scatter Plot: Crash Proximity to Fast-Food", fontsize=13)
plt.show()

# -------------------------------------------------------
# 6. Optional: Moran correlogram (spatial autocorrelation vs distance)
# -------------------------------------------------------
from esda.moran import Moran
import matplotlib.pyplot as plt

distances = [2, 4, 6, 8, 10]
morans = []
for k in distances:
    w_k = KNN.from_dataframe(gdf_crashes_m, k=k)
    w_k.transform = 'R'
    moran_k = Moran(gdf_crashes_m["fastfood_proximity"], w_k)
    morans.append(moran_k.I)

plt.figure(figsize=(6,4))
plt.plot(distances, morans, marker='o')
plt.title("Moran’s I vs Number of Neighbors (Spatial Correlogram)")
plt.xlabel("Number of Nearest Neighbors (k)")
plt.ylabel("Moran’s I")
plt.grid(True, alpha=0.3)
plt.show()


ModuleNotFoundError: No module named 'splot'

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from libpysal.weights import KNN

from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW
from mgwr.utils import shift_colormap

# -------------------------------------------------------
# 1. Load and prepare data
# -------------------------------------------------------
# Load crash and fast-food data
crashes = pd.read_csv("Data/CrashStatisticsFranklinCounty.csv").dropna(subset=["Latitude", "Longitude"])
fastfood = gpd.read_file("Data/fastfood_location_name_amenity.gpkg")

# Create GeoDataFrames
gdf_crashes = gpd.GeoDataFrame(
    crashes, geometry=gpd.points_from_xy(crashes["Longitude"], crashes["Latitude"]), crs="EPSG:4326"
)
gdf_fastfood = fastfood.to_crs(epsg=4326)

# Project to a local metric coordinate system (meters)
gdf_crashes_m = gdf_crashes.to_crs(epsg=3857)
gdf_fastfood_m = gdf_fastfood.to_crs(epsg=3857)

# -------------------------------------------------------
# 2. Create a grid or aggregate crash density spatially
# -------------------------------------------------------
# Create hexagonal grid or buffer zones to aggregate crashes
xmin, ymin, xmax, ymax = gdf_crashes_m.total_bounds
cell_size = 500  # 500 m grid

cols = np.arange(xmin, xmax + cell_size, cell_size)
rows = np.arange(ymin, ymax + cell_size, cell_size)

polygons = []
for x in cols:
    for y in rows:
        polygons.append(Point(x, y).buffer(cell_size / 2))

grid = gpd.GeoDataFrame(geometry=polygons, crs=gdf_crashes_m.crs)

# Count crashes in each grid cell
grid["crash_count"] = grid.geometry.apply(
    lambda g: gdf_crashes_m.within(g).sum()
)

# Count fast-food locations in each grid cell
grid["fastfood_count"] = grid.geometry.apply(
    lambda g: gdf_fastfood_m.within(g).sum()
)

# Filter cells with data
grid = grid[(grid["crash_count"] > 0) | (grid["fastfood_count"] > 0)].copy()

# -------------------------------------------------------
# 3. Prepare variables for GWR
# -------------------------------------------------------
# Dependent variable (crash density)
y = grid["crash_count"].values.reshape((-1, 1))

# Independent variable(s)
X = grid[["fastfood_count"]].values  # can add more columns if available

# Coordinates
u = grid.geometry.centroid.x.values
v = grid.geometry.centroid.y.values
coords = np.column_stack((u, v))

# -------------------------------------------------------
# 4. Bandwidth selection and model fitting
# -------------------------------------------------------
print("Selecting optimal bandwidth...")
bw = Sel_BW(coords, y, X).search(bw_min=2)
print("Optimal bandwidth:", bw)

# Fit the GWR model
gwr_model = GWR(coords, y, X, bw)
gwr_results = gwr_model.fit()

print(gwr_results.summary())

# -------------------------------------------------------
# 5. Attach results to the GeoDataFrame
# -------------------------------------------------------
grid["beta_fastfood"] = gwr_results.params[:, 1]
grid["beta_intercept"] = gwr_results.params[:, 0]
grid["local_r2"] = gwr_results.localR2

# -------------------------------------------------------
# 6. Visualize local coefficients (optional)
# -------------------------------------------------------
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
grid.plot(column="beta_fastfood", cmap="coolwarm", legend=True, ax=ax)
ax.set_title("Local GWR Coefficient for Fast-Food Count vs Crash Density")
ax.axis("off")
plt.show()



C:\Users\raab.75\AppData\Local\Temp\ipykernel_31036\1416954049.py:15: DtypeWarning: Columns (23,31) have mixed types. Specify dtype option on import or set low_memory=False.
  crashes = pd.read_csv("Data/CrashStatisticsFranklinCounty.csv").dropna(subset=["Latitude", "Longitude"])
